In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from pytorch3d.vis.plotly_vis import plot_scene
from data_tools import adv_dataset, adversarial_patch_3d

from attack_utils import *

CKPT_PATH = "output/train/PhysicalAdv/pointpillar/final_adversarial_patch_checkpoint.pt"

In [ ]:
state_dict = torch.load(CKPT_PATH)

In [ ]:
universal_adv_patch_car = single_sphere_legacy(scale=adv_dataset.CAR_ADV_PATCH_SCALE)
universal_adv_patch_car.load_parameter(state_dict["universal_adv_patch_car"])

print(state_dict["universal_adv_patch_car"])

In [ ]:
fig = plot_scene(
    {
        "original": {"mesh_1": universal_adv_patch_car.get_basic_meshes()},
        "adversarial": {"mesh_1": universal_adv_patch_car.get_transformed_meshes()},
    },
    ncols=2,
)
fig.update_layout(height=400, width=800)
fig.show()

In [ ]:
from pytorch3d.io import save_obj

output_meshes = universal_adv_patch_car.get_deformed_meshes()
save_obj(
    "PhysicalAdv_pointpillar.obj",
    output_meshes.verts_packed(),
    output_meshes.faces_packed(),
)

In [ ]:
from data_tools import simple_cubic_meshes, join_meshes_as_batch
from pytorch3d.vis.plotly_vis import plot_scene

test = simple_cubic_meshes(cubic_level=2)
meshes_batch = join_meshes_as_batch(test.get_deformed_lattice())

In [ ]:
fig = plot_scene({"original": {"mesh_1": meshes_batch}}, ncols=1)
fig.update_layout(height=400, width=400)
fig.show()

In [ ]:
import pandas as pd


def parse_table(data):
    # 将数据转换为DataFrame
    lines = data.strip().split("\n")
    rows = [line.split() for line in lines]
    columns = ["Category", "Value1", "Value2", "Value3", "Value4"]

    df = pd.DataFrame(rows, columns=columns)

    # 将数值列转换为浮点数
    df[["Value1", "Value2", "Value3", "Value4"]] = df[
        ["Value1", "Value2", "Value3", "Value4"]
    ].astype(float)

    print(df)

    clean_values = df.loc[
        df["Category"] == "Clean", ["Value1", "Value2", "Value3", "Value4"]
    ].values[0]

    # 计算每个类别相对于Clean的百分比
    percentage_df = df.copy()
    percentage_df[["Value1", "Value2", "Value3", "Value4"]] = (
        100 - df[["Value1", "Value2", "Value3", "Value4"]].div(clean_values) * 100
    )

    print("Original DataFrame:")
    print(df)
    print("\nPercentage DataFrame:")
    print(percentage_df)

In [ ]:
data_1 = """
Vanilla@BEV	76.9694	83.1086	88.1049	87.5583
Vanilla@3D	66.5986	66.2274	76.1004	73.9477
Car@BEV	76.8111	81.9277	87.8725	87.3614
Car@3D	65.8241	64.8201	75.0164	72.1968
"""

data_2 = """
Vanilla@BEV	76.9694	83.1086	88.1049	87.5583
Vanilla@3D	66.5986	66.2274	76.1004	73.9477
Car@BEV	76.8111	81.9277	87.8725	87.3614
Car@3D	65.8241	64.8201	75.0164	72.1968
"""


parse_table(data_1)

parse_table(data_2)

In [ ]:
import pandas as pd
import numpy as np


def parse_table(data):
    # 将数据转换为DataFrame
    lines = data.strip().split("\n")
    rows = [line.split() for line in lines[1:]]
    columns = lines[0].split()

    df = pd.DataFrame(rows, columns=columns)

    # 将数值列转换为浮点数
    df[columns[1:]] = df[columns[1:]].astype(float)

    print(df)

    return df

    # clean_values = df.loc[df['Category'] == 'Clean', ["Value1", "Value2", "Value3", "Value4"]].values[0]

    # # 计算每个类别相对于Clean的百分比
    # percentage_df = df.copy()
    # percentage_df[["Value1", "Value2", "Value3", "Value4"]] = 100 - df[["Value1", "Value2", "Value3", "Value4"]].div(clean_values) * 100

    # print("Original DataFrame:")
    # print(df)
    # print("\nPercentage DataFrame:")
    # print(percentage_df)


In [ ]:
table_1 = """
Metric	PointRCNN	PVRCNN	VoxelRCNN(Car)	Second
Clean@BEV	85.6744	88.9160	89.1513	88.7158
Clean@3D	78.6668	66.2274	76.1004	73.9477
Vanilla@BEV	76.9694	83.1086	88.1049	87.5583
Vanilla@3D	66.5986	66.2274	76.1004	73.9477
Car@BEV	76.8111	81.9277	87.8725	87.3614
Car@3D	65.8241	64.8201	75.0164	72.1968
"""
table_1: pd.DataFrame = parse_table(table_1)
clean_bev = table_1.loc[table_1["Metric"] == "Clean@BEV"].values[0, 1:].astype(float)
clean_3d = table_1.loc[table_1["Metric"] == "Clean@3D"].values[0, 1:].astype(float)
car_bev = table_1.loc[table_1["Metric"] == "Car@BEV"].values[0, 1:].astype(float)
car_3d = table_1.loc[table_1["Metric"] == "Car@3D"].values[0, 1:].astype(float)
# table_1 = table_1.append(new_row, ignore_index=True)
print(np.round((1 - (car_bev / clean_bev)) * 100, 2))
print(np.round((1 - (car_3d / clean_3d)) * 100, 2))

In [ ]:
table_1 = """
Metric	PointPIllar	PVRCNN	VoxelRCNN(Car)	Second
Clean@BEV	86.7564	88.9160	89.1513	88.7158
Clean@3D	77.5743	66.2274	76.1004	73.9477
Vanilla@BEV	80.6136	83.1086	88.1049	87.5583
Vanilla@3D	57.1795	66.2274	76.1004	73.9477
Car@BEV	79.3103	82.9085	82.7351	86.9980
Car@3D	53.3262	66.0200	65.8373	69.7959
"""
table_1: pd.DataFrame = parse_table(table_1)
clean_bev = table_1.loc[table_1["Metric"] == "Clean@BEV"].values[0, 1:].astype(float)
clean_3d = table_1.loc[table_1["Metric"] == "Clean@3D"].values[0, 1:].astype(float)
car_bev = table_1.loc[table_1["Metric"] == "Car@BEV"].values[0, 1:].astype(float)
car_3d = table_1.loc[table_1["Metric"] == "Car@3D"].values[0, 1:].astype(float)
# table_1 = table_1.append(new_row, ignore_index=True)
print(np.round((1 - (car_bev / clean_bev)) * 100, 2))
print(np.round((1 - (car_3d / clean_3d)) * 100, 2))

In [ ]:
table_1 = """
Metric	PointPIllar	PVRCNN	VoxelRCNN(Car)	Second
Clean@BEV	86.7564	88.9160	89.1513	88.7158
Clean@3D	77.5743	66.2274	76.1004	73.9477
Vanilla@BEV	80.6136	83.1086	88.1049	87.5583
Vanilla@3D	57.1795	66.2274	76.1004	73.9477
Car@BEV	80.0401	82.2935	87.8767	87.2270
Car@3D	56.2568	64.8246	74.1950	70.3205
"""
table_1: pd.DataFrame = parse_table(table_1)
clean_bev = table_1.loc[table_1["Metric"] == "Clean@BEV"].values[0, 1:].astype(float)
clean_3d = table_1.loc[table_1["Metric"] == "Clean@3D"].values[0, 1:].astype(float)
car_bev = table_1.loc[table_1["Metric"] == "Car@BEV"].values[0, 1:].astype(float)
car_3d = table_1.loc[table_1["Metric"] == "Car@3D"].values[0, 1:].astype(float)
# table_1 = table_1.append(new_row, ignore_index=True)
print(np.round((1 - (car_bev / clean_bev)) * 100, 2))
print(np.round((1 - (car_3d / clean_3d)) * 100, 2))

In [ ]:
table_1 = """
Metric	PointPIllar		VoxelRCNN(Car)	Second	PointRCNN
Clean@BEV	86.7564	89.1513	88.7158	85.6744
Clean@3D	77.5743	76.1004	73.9477	78.6668
Vanilla@BEV	80.6136		88.1049	87.5583	76.9694
Vanilla@3D	57.1795		76.1004	73.9477	66.5986
Car@BEV	79.3931		88.0110	86.8020	76.7037
Car@3D	54.5816		75.2899	70.1040	65.7802
"""
table_1: pd.DataFrame = parse_table(table_1)
clean_bev = table_1.loc[table_1["Metric"] == "Clean@BEV"].values[0, 1:].astype(float)
clean_3d = table_1.loc[table_1["Metric"] == "Clean@3D"].values[0, 1:].astype(float)
car_bev = table_1.loc[table_1["Metric"] == "Car@BEV"].values[0, 1:].astype(float)
car_3d = table_1.loc[table_1["Metric"] == "Car@3D"].values[0, 1:].astype(float)
# table_1 = table_1.append(new_row, ignore_index=True)
print(np.round((1 - (car_bev / clean_bev)) * 100, 2))
print(np.round((1 - (car_3d / clean_3d)) * 100, 2))

In [ ]:
import os
import re
import json
from collections import defaultdict


STEP_ABLATION_ROOT_PATH = "output/train/step_ablation"


def extract_info_from_folder(folder_name):
    # 使用正则表达式提取信息
    match = re.match(r"([a-zA-Z_]+)_([a-zA-Z]+)_([\d\.]+)", folder_name)
    if match:
        model_name = match.group(1)
        attack_method = match.group(2)
        parameter = match.group(3)
        return model_name, attack_method, parameter
    return None


def read_evaluation_result_json(folder_path):
    json_path = os.path.join(folder_path, "evaluation_result.json")
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            return json.load(f)
    return None


def get_folder_info_by_model(directory):
    model_dict = defaultdict(lambda: defaultdict(dict))

    # 遍历指定目录中的文件夹
    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)

        if os.path.isdir(folder_path):
            extracted_info = extract_info_from_folder(folder_name)
            if extracted_info:
                model_name, attack_method, parameter = extracted_info

                # 读取 evaluation_result.json 文件
                evaluation_result = read_evaluation_result_json(folder_path)

                # 按照模型名称和参数归类
                model_dict[model_name][parameter] = {
                    "path": folder_path,
                    "attack_method": attack_method,
                    "evaluation_result": evaluation_result,
                }

    return model_dict


model_dict = get_folder_info_by_model(STEP_ABLATION_ROOT_PATH)


model_graph = {}
# 打印结果
for model_name, parameters in model_dict.items():
    print(f"Model: {model_name}")
    model_graph[model_name] = {"car_bev": [], "car_3d": []}

    for step_size in ["0.5", "0.05", "0.005", "0.0005", "0.00005"]:
        info = parameters[step_size]

        car_3d = info["evaluation_result"]["Car_3d/moderate"]
        car_bev = info["evaluation_result"]["Car_bev/moderate"]
        model_graph[model_name]["car_bev"].append(car_bev)
        model_graph[model_name]["car_3d"].append(car_3d)

    print(f"length of car_bev:{model_graph[model_name]['car_bev'].__len__()}")
    print(f"length of car_3d:{model_graph[model_name]['car_3d'].__len__()}")
    # print(f"  Info: {info}")
print(model_graph)

In [ ]:
import pandas as pd

# import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd


def parse_table(data):
    # 将数据转换为DataFrame
    lines = data.strip().split("\n")
    rows = [line.split() for line in lines[1:]]
    columns = lines[0].split()

    df = pd.DataFrame(rows, columns=columns)

    # 将数值列转换为浮点数
    df[columns[1:]] = df[columns[1:]].astype(float)

    return df


table_1 = """
Model@lr	PointRCNN@0.5	PointRCNN@0.05	PointRCNN@0.005	PointRCNN@0.0005	PointRCNN@0.00005
Car@BEV(IFGSM)	73.7186	73.9964	76.4810	77.0568	76.9188
Car@3D(IFGSM)	62.5936	62.2509	65.4640	65.9328	66.0451
Car@BEV(Adam)	63.5294	64.1308	72.4848	76.4491	77.0371
Car@3D(Adam)	50.8117	50.6058	61.0131	65.5077	65.8303
"""

table_1 = parse_table(table_1)

table_2 = """
Model@lr	PointPillar@0.5	PointPillar@0.05	PointPillar@0.005	PointPillar@0.0005	PointPillar@0.00005
Car@BEV(IFGSM)	74.7179	76.7819	79.5679	80.3516	80.7451
Car@3D(IFGSM)	42.3016	46.0600	54.0880	56.7081	57.2705
Car@BEV(Adam)	74.8059	75.7801	79.2733	80.2302	80.9181
Car@3D(Adam)	49.7496	48.5106	56.7255	57.5994	57.2783
"""

table_2 = parse_table(table_2)

table_3 = """
Model@lr	PVRCNN@0.5	PVRCNN@0.05	PVRCNN@0.005	PVRCNN@0.0005	PVRCNN@0.00005
Car@BEV(IFGSM)	81.8204	81.6676	75.4499	80.7042	81.0519
Car@3D(IFGSM)	58.3808	62.2472	52.8814	57.1060	57.8906
Car@BEV(Adam)	82.4894	82.6652	81.6014	81.0678	81.0859
Car@3D(Adam)	63.4156	63.1228	59.8393	58.3011	57.9622
"""

table_3 = parse_table(table_3)

table_4 = """
Model@lr	VoxelRCNN@0.5	VoxelRCNN@0.05	VoxelRCNN@0.005	VoxelRCNN@0.0005	VoxelRCNN@0.00005
Car@BEV(IFGSM)	75.6491	75.7126	76.3095	75.8909	76.0998
Car@3D(IFGSM)	64.6322	64.5856	64.4498	59.4109	59.8451
Car@BEV(Adam)	76.3462	77.4697	77.4458	75.7331	76.1522
Car@3D(Adam)	64.5041	66.4997	67.6613	60.2458	59.9528
"""

table_4 = parse_table(table_4)

table_5 = """
Model@lr	SECOND@0.5	SECOND@0.05	SECOND@0.005	SECOND@0.0005	SECOND@0.00005
Car@BEV(IFGSM)	83.3245	82.6507	83.6077	81.6649	81.3468
Car@3D(IFGSM)	71.0220	71.1782	73.5252	66.5171	66.3696
Car@BEV(Adam)	86.3404	86.6090	85.5537	82.0886	81.6318
Car@3D(Adam)	76.1328	76.3723	72.1165	67.4521	66.8208
"""

table_5 = parse_table(table_5)


In [ ]:
import seaborn as sns


def parse_pd(table):
    model_name = table.columns[1].split("@")[0]
    car_bev = table.iloc[2][1:].to_numpy().astype(float)
    car_3d = table.iloc[3][1:].to_numpy().astype(float)
    print(table.iloc[2][0])
    print(table.iloc[3][0])
    print(model_name)

    return car_bev, car_3d


def parse_graph(car_bev_ifgsm, car_3d_ifgsm, car_bev_adam, car_3d_adam, title: str):
    data = {
        "step size/learning rate": [0.5, 0.05, 0.005, 0.0005, 0.00005],
        "Car@BEV(IFGSM)": car_bev_ifgsm,
        "Car@3D(IFGSM)": car_3d_ifgsm,
        "Car@BEV(Adam)": car_bev_adam,
        "Car@3D(Adam)": car_3d_adam,
    }

    df = pd.DataFrame(data)

    # 将数据转换为长格式（long-form），以便使用 seaborn 进行可视化
    df_melted = df.melt(
        "step size/learning rate", var_name="Metric", value_name="Value"
    )

    # 绘制折线图
    sns.set(style="whitegrid")
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=df_melted, x="step size/learning rate", y="Value", hue="Metric", marker="o"
    )

    # 设置图表标题和轴标签
    plt.title("Performance Metrics vs. Step Size/Learning Rate")
    plt.xlabel("Step Size/Learning Rate")
    plt.ylabel("Performance Metric")
    plt.title(title)
    plt.xscale("log")

    # 显示图例
    plt.legend(title="Metric")

    # 显示图表
    plt.show()
    # 设置Seaborn样式

    # sns.set(style="whitegrid")
    # sns.lineplot(x='step size/learning rate', y=df.loc[1].values[0], data=df, marker='o', label=df.loc[1].values[0], color='blue', linestyle='-')
    # sns.lineplot(x='step size/learning rate', y=df.loc[2].values[0], data=df, marker='o', label=df.loc[2].values[0], color='blue', linestyle='--')
    # sns.lineplot(x='step size/learning rate', y=df.loc[3].values[0], data=df, marker='o', label=df.loc[3].values[0], color='red', linestyle='-')
    # sns.lineplot(x='step size/learning rate', y=df.loc[4].values[0], data=df, marker='o', label=df.loc[4].values[0], color='red', linestyle='--')
    # plt.title(title)
    # plt.xscale('log')
    # plt.legend()
    # plt.show()


car_bev_ifgsm, car_3d_ifgsm = (
    model_graph["pointrcnn"]["car_bev"],
    model_graph["pointrcnn"]["car_3d"],
)
car_bev_adam, car_3d_adam = parse_pd(table_1)
for step, y in zip(["0.5", "0.05", "0.005", "0.0005", "0.00005"], car_bev_ifgsm):
    print(f"({step},{y:.2f})", end=" ")
print()
for step, y in zip(["0.5", "0.05", "0.005", "0.0005", "0.00005"], car_3d_ifgsm):
    print(f"({step},{y:.2f})", end=" ")

parse_graph(car_bev_ifgsm, car_3d_ifgsm, car_bev_adam, car_3d_adam, "PointRCNN")

car_bev_ifgsm, car_3d_ifgsm = (
    model_graph["pointpillar"]["car_bev"],
    model_graph["pointpillar"]["car_3d"],
)
car_bev_adam, car_3d_adam = parse_pd(table_2)
for step, y in zip(["0.5", "0.05", "0.005", "0.0005", "0.00005"], car_bev_ifgsm):
    print(f"({step},{y:.2f})", end=" ")
print()
for step, y in zip(["0.5", "0.05", "0.005", "0.0005", "0.00005"], car_3d_ifgsm):
    print(f"({step},{y:.2f})", end=" ")
parse_graph(car_bev_ifgsm, car_3d_ifgsm, car_bev_adam, car_3d_adam, "PointPillar")

car_bev_ifgsm, car_3d_ifgsm = (
    model_graph["pvrcnn"]["car_bev"],
    model_graph["pvrcnn"]["car_3d"],
)
car_bev_adam, car_3d_adam = parse_pd(table_3)
for step, y in zip(["0.5", "0.05", "0.005", "0.0005", "0.00005"], car_bev_ifgsm):
    print(f"({step},{y:.2f})", end=" ")
print()
for step, y in zip(["0.5", "0.05", "0.005", "0.0005", "0.00005"], car_3d_ifgsm):
    print(f"({step},{y:.2f})", end=" ")
parse_graph(car_bev_ifgsm, car_3d_ifgsm, car_bev_adam, car_3d_adam, "PVRCNN")


car_bev_ifgsm, car_3d_ifgsm = (
    model_graph["voxel_rcnn_car"]["car_bev"],
    model_graph["voxel_rcnn_car"]["car_3d"],
)
car_bev_adam, car_3d_adam = parse_pd(table_4)
for step, y in zip(["0.5", "0.05", "0.005", "0.0005", "0.00005"], car_bev_ifgsm):
    print(f"({step},{y:.2f})", end=" ")
print()
for step, y in zip(["0.5", "0.05", "0.005", "0.0005", "0.00005"], car_3d_ifgsm):
    print(f"({step},{y:.2f})", end=" ")
parse_graph(car_bev_ifgsm, car_3d_ifgsm, car_bev_adam, car_3d_adam, "VoxelRCNN")

car_bev_ifgsm, car_3d_ifgsm = (
    model_graph["second"]["car_bev"],
    model_graph["second"]["car_3d"],
)
car_bev_adam, car_3d_adam = parse_pd(table_5)
for step, y in zip(["0.5", "0.05", "0.005", "0.0005", "0.00005"], car_bev_ifgsm):
    print(f"({step},{y:.2f})", end=" ")
print()
for step, y in zip(["0.5", "0.05", "0.005", "0.0005", "0.00005"], car_3d_ifgsm):
    print(f"({step},{y:.2f})", end=" ")
parse_graph(car_bev_ifgsm, car_3d_ifgsm, car_bev_adam, car_3d_adam, "SECOND")


parse_pd(table_5)


# table_1.iloc[2][1:].to_numpy().astype(float)
# table_1.iloc[2][1:].to_numpy().astype(float)
# table_1.columns[1]


PointRCNN
(0.05,66.23)
(0.05,52.72)

PointPillar
(0.005,66.29)
(0.005,43.09)

PVRCNN
(0.0005,81.08)
(0.0005,58.31) 

VoxelRCNN
(0.0005,75.72) 
(0.0005,60.16)

SECOND
(0.0005,81.83)
(0.0005,67.74)

In [ ]:
import os
import re
import json
from collections import defaultdict


STEP_ABLATION_ROOT_PATH = "output/train/scale_ablation"
LEVEL_ABLATION_ROOT_PATH = "output/train/level_ablation"

# PointRCNN PointPillar PVRCNN VoxelRCNN SECOND
NORMAL_STEP = ["0.05", "0.005", "0.0005", "0.0005", "0.0005"]

NORMAL_SCALE_ = [81.6318]


def extract_info_from_folder(folder_name):
    # 使用正则表达式提取信息
    match = re.match(r"([a-zA-Z_\d]+)_([\d.]+)_([\d.]+)_([\d.]+)", folder_name)
    if match:
        attack_method = match.group(1)
        scale_1 = match.group(2)
        return attack_method, scale_1
    return None


def read_evaluation_result_json(folder_path):
    json_path = os.path.join(folder_path, "evaluation_result.json")
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            return json.load(f)
    return None


def get_folder_info_by_model(directory):
    model_dict = defaultdict(lambda: defaultdict(dict))

    # 遍历指定目录中的文件夹
    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)

        if os.path.isdir(folder_path):
            extracted_info = extract_info_from_folder(folder_name)
            if extracted_info:
                attack_method, parameter = extracted_info
                # 读取 evaluation_result.json 文件
                evaluation_result = read_evaluation_result_json(folder_path)

                # 按照模型名称和参数归类
                model_dict[attack_method][parameter] = {
                    "evaluation_result": evaluation_result
                }

    return model_dict


model_dict = get_folder_info_by_model(STEP_ABLATION_ROOT_PATH)

# print(model_dict)

model_graph = {}
# 打印结果
for model_name, parameters in model_dict.items():
    # print(f"Model: {model_name}")
    model_graph[model_name] = {"car_bev": [], "car_3d": []}

    for step_size in ["0.5", "0.6", "0.8", "0.9", "1.0"]:
        info = parameters[step_size]

        if info["evaluation_result"] is None:
            continue
        car_3d = info["evaluation_result"]["Car_3d/moderate"]
        car_bev = info["evaluation_result"]["Car_bev/moderate"]
        model_graph[model_name]["car_bev"].append(car_bev)
        model_graph[model_name]["car_3d"].append(car_3d)

    # print(f"length of car_bev:{model_graph[model_name]['car_bev'].__len__()}")
    # print(f"length of car_3d:{model_graph[model_name]['car_3d'].__len__()}")
    # print(f"  Info: {info}")
print(model_graph)

for attack_methods in model_graph.keys():
    if model_graph[attack_methods]["car_bev"].__len__() == 0:
        continue

    print(attack_methods)
    print("BEV:")
    for i, step_size in enumerate(["0.5", "0.6", "0.8", "0.9", "1.0"]):
        car_bev = model_graph[attack_methods]["car_bev"][i]
        print(f"({step_size}, {car_bev:.2f})", end=" ")

    print()
    print("3D:")
    for i, step_size in enumerate(["0.5", "0.6", "0.8", "0.9", "1.0"]):
        car_3d = model_graph[attack_methods]["car_3d"][i]
        print(f"({step_size}, {car_3d:.2f})", end=" ")

    print()

In [ ]:
import os
import re
import json
from collections import defaultdict


LEVEL_ABLATION_ROOT_PATH = "output/train/level_ablation"

# PointRCNN PointPillar PVRCNN VoxelRCNN SECOND
NORMAL_STEP = ["0.05", "0.005", "0.0005", "0.0005", "0.0005"]

NORMAL_SCALE_ = [81.6318]


def extract_info_from_folder(folder_name):
    # 使用正则表达式提取信息
    match = re.match(r"([a-zA-Z_\d]+)_([\d]+)", folder_name)
    if match:
        attack_method = match.group(1)
        level = match.group(2)
        return attack_method, level
    return None


def read_evaluation_result_json(folder_path):
    json_path = os.path.join(folder_path, "evaluation_result.json")
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            return json.load(f)
    return None


def get_folder_info_by_model(directory):
    model_dict = defaultdict(lambda: defaultdict(dict))

    # 遍历指定目录中的文件夹
    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)

        if os.path.isdir(folder_path):
            extracted_info = extract_info_from_folder(folder_name)
            if extracted_info:
                attack_method, parameter = extracted_info
                # 读取 evaluation_result.json 文件
                evaluation_result = read_evaluation_result_json(folder_path)

                # 按照模型名称和参数归类
                model_dict[attack_method][parameter] = {
                    "evaluation_result": evaluation_result
                }

    return model_dict


model_dict = get_folder_info_by_model(LEVEL_ABLATION_ROOT_PATH)

# print(model_dict)
print(model_dict)
model_graph = {}
# 打印结果
for model_name, parameters in model_dict.items():
    model_graph[model_name] = {"car_bev": [], "car_3d": []}

    for level in ["0", "1", "3"]:
        info = parameters[level]
        if info["evaluation_result"] is None:
            continue
        car_3d = info["evaluation_result"]["Car_3d/moderate"]
        car_bev = info["evaluation_result"]["Car_bev/moderate"]
        model_graph[model_name]["car_bev"].append(car_bev)
        model_graph[model_name]["car_3d"].append(car_3d)

    # print(f"length of car_bev:{model_graph[model_name]['car_bev'].__len__()}")
    # print(f"length of car_3d:{model_graph[model_name]['car_3d'].__len__()}")
    # print(f"  Info: {info}")
print(model_graph)

for attack_methods in model_graph.keys():
    if model_graph[attack_methods]["car_bev"].__len__() == 0:
        continue

    print(attack_methods)
    print("BEV:")
    for i, step_size in enumerate(["0", "1", "3"]):
        car_bev = model_graph[attack_methods]["car_bev"][i]
        print(f"({step_size}, {car_bev:.2f})", end=" ")

    print()
    print("3D:")
    for i, step_size in enumerate(["0", "1", "3"]):
        car_3d = model_graph[attack_methods]["car_3d"][i]
        print(f"({step_size}, {car_3d:.2f})", end=" ")

    print()

In [ ]:
import os
import json
from copy import deepcopy


def load_evaluation_results(root_dir):
    results = {}

    # 遍历模型名称文件夹
    for model_name in os.listdir(root_dir):
        model_path = os.path.join(root_dir, model_name)
        if os.path.isdir(model_path):
            results[model_name] = {}

            # 遍历攻击方法文件夹
            for attack_method in os.listdir(model_path):
                attack_path = os.path.join(model_path, attack_method)
                if os.path.isdir(attack_path):
                    evaluation_file = os.path.join(
                        attack_path, "evaluation_result.json"
                    )

                    # 读取evaluation_result.json文件
                    if os.path.exists(evaluation_file):
                        with open(evaluation_file, "r") as f:
                            try:
                                evaluation_data = json.load(f)
                                results[model_name][attack_method] = evaluation_data
                            except json.JSONDecodeError:
                                print(f"Error decoding JSON in file: {evaluation_file}")

    return results


# 使用示例
root_directory = "output/train/loss_ablation"
evaluation_results = load_evaluation_results(root_directory)
evaluation_results_all = deepcopy(evaluation_results)
print(evaluation_results.keys())
# 打印结果
# print(json.dumps(evaluation_results, indent=4))
clean_data = [86.76, 77.57, 85.67, 78.67, 85.67, 78.67, 89.15, 85.66, 89.15, 85.66]
for attack_method in [
    "mislocalize_4",
    "mislocalize_7",
    "mislocalize_9",
    "misrecognize_4",
    "misrecognize_8",
    "misrecognize_9",
    "misrecognize_10",
    "comprehensive_4",
    "comprehensive_9",
]:
    for idx, model_name in enumerate(
        [
            "pointpillar",
            "pointrcnn_s1",
            "pointrcnn",
            "voxel_rcnn_car_s1",
            "voxel_rcnn_car",
        ]
    ):
        clean_bev = clean_data[idx * 2]
        clean_3d = clean_data[idx * 2 + 1]
        car_bev = (
            (
                clean_bev
                - evaluation_results[model_name][attack_method]["Car_bev/moderate"]
            )
            / clean_bev
            * 100
        )
        car_3d = (
            (
                clean_3d
                - evaluation_results[model_name][attack_method]["Car_3d/moderate"]
            )
            / clean_3d
            * 100
        )
        print(f",\t{car_bev:.2f}\% ,\t{car_3d:.2f}\% ", end="")
    print()

In [ ]:
import os
import json


def load_model_results(root_dir):
    results = {}

    # 遍历模型名称文件夹
    for model_name in os.listdir(root_dir):
        model_path = os.path.join(root_dir, model_name)
        if os.path.isdir(model_path):
            evaluation_file = os.path.join(model_path, "evaluation_result.json")

            # 读取evaluation_result.json文件
            if os.path.exists(evaluation_file):
                with open(evaluation_file, "r") as f:
                    try:
                        evaluation_data = json.load(f)
                        results[model_name] = evaluation_data
                    except json.JSONDecodeError:
                        print(f"Error decoding JSON in file: {evaluation_file}")

    return results


# 使用示例
root_directory = "output/train/PhysicalAdv"
model_results = load_model_results(root_directory)

print(model_results.keys())
clean_data = [86.76, 77.57, 85.67, 78.67, 88.92, 84.37, 89.15, 85.66, 88.72, 88.72]
for i, model_name in enumerate(
    ["pointpillar", "pointrcnn", "pvrcnn", "voxel_rcnn_car", "second"]
):
    model_metrics = model_results[model_name]
    clean_bev = clean_data[2 * i]
    clean_3d = clean_data[2 * i + 1]
    car_bev = (clean_bev - model_metrics["Car_bev/moderate"]) / clean_bev * 100
    car_3d = (clean_3d - model_metrics["Car_3d/moderate"]) / clean_3d * 100
    print(f"&\t{car_bev:.2f}\% &\t{car_3d:.2f}\% ", end="")
print()
# 打印结果
# print(json.dumps(model_results, indent=4))


In [ ]:
import os
import json


def load_evaluation_results(root_dir):
    results = {}

    # 遍历模型名称文件夹
    for model_name in os.listdir(root_dir):
        model_path = os.path.join(root_dir, model_name)
        if os.path.isdir(model_path):
            results[model_name] = {}

            # 遍历攻击方法文件夹
            for attack_method in os.listdir(model_path):
                attack_path = os.path.join(model_path, attack_method)
                if os.path.isdir(attack_path):
                    evaluation_file = os.path.join(
                        attack_path, "evaluation_result.json"
                    )

                    # 读取evaluation_result.json文件
                    if os.path.exists(evaluation_file):
                        with open(evaluation_file, "r") as f:
                            try:
                                evaluation_data = json.load(f)
                                results[model_name][attack_method] = evaluation_data
                            except json.JSONDecodeError:
                                print(f"Error decoding JSON in file: {evaluation_file}")

    return results


# 使用示例
root_directory = "output/train/comparison_table"
evaluation_results = load_evaluation_results(root_directory)

print(json.dumps(evaluation_results, indent=4))
clean_data = [86.76, 77.57, 85.67, 78.67, 88.92, 84.37, 89.15, 85.66, 88.72, 81.82]
for attack_method in ["misrecognize_10", "misrecognize_9"]:
    for i, model_name in enumerate(
        ["pointpillar", "pointrcnn", "pvrcnn", "voxel_rcnn_car", "second"]
    ):
        model_metrics = evaluation_results[model_name][attack_method]
        if model_name in ["pointpillar", "pointrcnn", "voxel_rcnn_car"]:
            model_metrics = evaluation_results_all[model_name][attack_method]
        clean_bev = clean_data[2 * i]
        clean_3d = clean_data[2 * i + 1]
        car_bev = (clean_bev - model_metrics["Car_bev/moderate"]) / clean_bev * 100
        car_3d = (clean_3d - model_metrics["Car_3d/moderate"]) / clean_3d * 100
        print(f"&\t{car_bev:.2f}\% &\t{car_3d:.2f}\% ", end="")
    print()
    # 打印结果
    # print(json.dumps(evaluation_results, indent=4))


In [ ]:
import os
import re
import json
from collections import defaultdict


LEVEL_ABLATION_ROOT_PATH = "output/train/transfer_eval"

# PointRCNN PointPillar PVRCNN VoxelRCNN SECOND
clean_data = [86.76, 77.57, 85.67, 78.67, 88.92, 84.37, 89.15, 85.66, 88.72, 81.82]


def extract_info_from_folder(folder_name):
    # 使用正则表达式提取信息
    match = re.match(r"([a-zA-Z_1]+)_([a-zA-Z]+_[\d]+)", folder_name)
    if match:
        surrogate_model = match.group(1)
        attack_method = match.group(2)
        return surrogate_model, attack_method
    return None


def read_evaluation_result_json(folder_path):
    json_path = os.path.join(folder_path, "evaluation_result.json")
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            return json.load(f)
    return None


def get_folder_info_by_model(directory):
    model_dict = defaultdict(lambda: defaultdict(dict))

    # 遍历指定目录中的文件夹
    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)
        model_name = folder_name
        for sub_folder_name in os.listdir(folder_path):
            sub_folder_path = os.path.join(folder_path, sub_folder_name)
            if os.path.isdir(sub_folder_path):
                extracted_info = extract_info_from_folder(sub_folder_name)
                print(extracted_info)
            if extracted_info:
                surrogate_model, attack_method = extracted_info
                # 读取 evaluation_result.json 文件
                evaluation_result = read_evaluation_result_json(sub_folder_path)
                if evaluation_result is None:
                    continue
                # 按照模型名称和参数归类
                model_dict[model_name][surrogate_model][attack_method] = {
                    "evaluation_result": evaluation_result
                }

    return model_dict


model_dict = get_folder_info_by_model(LEVEL_ABLATION_ROOT_PATH)
print(json.dumps(model_dict, indent=4))


for surrogate_model in [
    "pointpillar",
    "pointrcnn_s1",
    "pointrcnn",
    "voxel_rcnn_car_s1",
    "voxel_rcnn_car",
]:
    for attack_method in ["misrecognize_9", "misrecognize_10"]:
        for i, victim_model in enumerate(
            ["pointpillar", "pointrcnn", "pvrcnn", "voxel_rcnn_car", "second"]
        ):
            # print("BEV:")
            if attack_method not in model_dict[victim_model][surrogate_model]:
                print(f"& \t-& \t- ", end=" ")
            else:
                clean_bev = clean_data[2 * i]
                clean_3d = clean_data[2 * i + 1]

                car_bev = (
                    (
                        clean_bev
                        - model_dict[victim_model][surrogate_model][attack_method][
                            "evaluation_result"
                        ]["Car_bev/moderate"]
                    )
                    / clean_bev
                    * 100
                )
                car_3d = (
                    (
                        clean_3d
                        - model_dict[victim_model][surrogate_model][attack_method][
                            "evaluation_result"
                        ]["Car_3d/moderate"]
                    )
                    / clean_3d
                    * 100
                )
                print(f"& \t{car_bev:.2f}\%& \t{car_3d:.2f}\% ", end=" ")

        print()

In [ ]:
import os
import re
import json
from collections import defaultdict

# PointRCNN PointPillar PVRCNN VoxelRCNN SECOND
clean_data = [86.76, 77.57, 85.67, 78.67, 88.92, 84.37, 89.15, 85.66, 88.72, 81.82]


def extract_info_from_folder(folder_name):
    # 使用正则表达式提取信息
    match = re.match(
        r"query_attack_(?P<model>[a-zA-Z_]+)_(?P<surrogate_model>pr_s1|pv_s1|pr|pv|pp)",
        folder_name,
    )
    if match:
        model = match.group("model")
        surrogate_model = match.group("surrogate_model")
        return model, surrogate_model
    return None


def read_evaluation_result_json(folder_path):
    json_path = os.path.join(folder_path, "evaluation_result.json")
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            return json.load(f)
    return None


def get_folder_info_by_model(directory):
    model_dict = defaultdict(lambda: defaultdict(dict))

    # 遍历指定目录中的文件夹
    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)

        if os.path.isdir(folder_path):
            extracted_info = extract_info_from_folder(folder_name)

            if extracted_info is None:
                print(folder_name)
            else:
                print(extracted_info)
        if extracted_info:
            model, surrogate_model = extracted_info
            # 读取 evaluation_result.json 文件
            evaluation_result = read_evaluation_result_json(folder_path)
            if evaluation_result is None:
                continue
            # 按照模型名称和参数归类
            model_dict[model][surrogate_model] = {
                "evaluation_result": evaluation_result
            }

    return model_dict


def print_keys(d, level=0, max_level=-1):
    # 遍历字典的每一个 key-value 对
    if level == max_level:
        return
    for key, value in d.items():
        # 打印当前 key 和它所在的层次 level
        print("  " * level + str(key))

        # 如果 value 是一个字典，递归调用自身
        if isinstance(value, dict):
            print_keys(value, level + 1, max_level)


model_dict = get_folder_info_by_model("output/train/query_attack")
# print(model_dict.keys())
# print(json.dumps(model_dict, indent=4))
print_keys(model_dict, max_level=2)

for surrogate_model in ["pp", "pv_s1", "pv"]:
    for i, victim_model in enumerate(
        ["pointpillar", "pointrcnn", "pvrcnn", "voxel_rcnn", "second"]
    ):
        if surrogate_model not in model_dict[victim_model]:
            print(f"& \t-& \t- ", end=" ")
        else:
            clean_bev = clean_data[2 * i]
            clean_3d = clean_data[2 * i + 1]

            car_bev = model_dict[victim_model][surrogate_model]["evaluation_result"][
                "Car_bev/moderate"
            ]
            car_3d = model_dict[victim_model][surrogate_model]["evaluation_result"][
                "Car_3d/moderate"
            ]
            print(f"& \t{car_bev:.2f}& \t{car_3d:.2f} ", end=" ")

    print()

In [ ]:
import os
import re
import json
from collections import defaultdict

# PointRCNN PointPillar PVRCNN VoxelRCNN SECOND
clean_data = [86.76, 77.57, 85.67, 78.67, 88.92, 84.37, 89.15, 85.66, 88.72, 81.82]
# pattern = """Car AP@0.70, 0.70, 0.70:
# bbox AP:90.3303, 81.2013, 80.8667
# bev  AP:87.4439, 83.2445, 82.4719
# 3d   AP:81.4915, 66.3580, 65.6103
# aos  AP:90.25, 80.95, 80.53"""
pattern = r"""Car AP@0.70, 0.70, 0.70:
bbox AP:(?P<bbox>\d+.\d+, \d+.\d+, \d+.\d+)
bev  AP:(?P<bev>\d+.\d+, \d+.\d+, \d+.\d+)
3d   AP:(?P<threed>\d+.\d+, \d+.\d+, \d+.\d+)
aos  AP:(?P<aos>\d+.\d+, \d+.\d+, \d+.\d+)"""


def extract_info_from_folder(folder_name):
    # 使用正则表达式提取信息
    match = re.match(
        r"query_attack_(?P<model>[a-zA-Z_]+)_(?P<surrogate_model>pr_s1|pv_s1|vr_s1|vr|pr|pv|pp)",
        folder_name,
    )
    if match:
        model = match.group("model")
        surrogate_model = match.group("surrogate_model")
        return model, surrogate_model
    return None


def read_evaluation_result_json(folder_path):
    json_path = os.path.join(folder_path, "exp_log.log")
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            return "".join(f.readlines())
    return None


def get_folder_info_by_model(directory):
    model_dict = defaultdict(lambda: defaultdict(dict))

    # 遍历指定目录中的文件夹
    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)

        if os.path.isdir(folder_path):
            extracted_info = extract_info_from_folder(folder_name)

            if extracted_info is None:
                print(folder_name)
            else:
                print(extracted_info)
        if extracted_info:
            model, surrogate_model = extracted_info
            # 读取 evaluation_result.json 文件
            evaluation_result = read_evaluation_result_json(folder_path)
            if evaluation_result is None:
                continue
            match = re.search(pattern, evaluation_result)
            bev = [float(num) for num in match.group("bev").strip().split(",")]
            threed = [float(num) for num in match.group("threed").strip().split(",")]
            # 按照模型名称和参数归类
            model_dict[model][surrogate_model] = {"bev": bev[1], "3d": threed[1]}

    return model_dict


def print_keys(d, level=0, max_level=-1):
    # 遍历字典的每一个 key-value 对
    if level == max_level:
        return
    for key, value in d.items():
        # 打印当前 key 和它所在的层次 level
        print("  " * level + str(key))

        # 如果 value 是一个字典，递归调用自身
        if isinstance(value, dict):
            print_keys(value, level + 1, max_level)


model_dict = get_folder_info_by_model("output/train/query_attack")
# print(model_dict.keys())
# print(json.dumps(model_dict, indent=4))
print_keys(model_dict, max_level=2)
print(model_dict)
for surrogate_model in ["pp", "pr_s1", "pr", "pv_s1", "pv", "vr_s1", "vr"]:
    for i, victim_model in enumerate(
        ["pointpillar", "pointrcnn", "pvrcnn", "voxel_rcnn", "second"]
    ):
        if surrogate_model not in model_dict[victim_model]:
            print(f"& \t-& \t- ", end=" ")
        else:
            clean_bev = clean_data[2 * i]
            clean_3d = clean_data[2 * i + 1]

            car_bev = (
                (clean_bev - model_dict[victim_model][surrogate_model]["bev"])
                / clean_bev
                * 100
            )
            car_3d = (
                (clean_3d - model_dict[victim_model][surrogate_model]["3d"])
                / clean_3d
                * 100
            )
            print(f"& \t{car_bev:.2f}\%& \t{car_3d:.2f}\% ", end=" ")

    print()

In [ ]:
import re

text = """sdadadCar AP@0.70, 0.70, 0.70:
bbox AP:70.1641, 66.5456, 65.3719
bev  AP:74.6308, 71.5723, 68.1586
3d   AP:44.1355, 40.7902, 39.7297
aos  AP:69.82, 65.58, 64.26"""

# 正则表达式模式
pattern = r"""Car AP@0.70, 0.70, 0.70:
bbox AP:(?P<bbox>\d+.\d+, \d+.\d+, \d+.\d+)
bev  AP:(?P<bev>\d+.\d+, \d+.\d+, \d+.\d+)
3d   AP:(?P<threed>\d+.\d+, \d+.\d+, \d+.\d+)
aos  AP:(?P<aos>\d+.\d+, \d+.\d+, \d+.\d+)"""

# 使用 re.VERBOSE 使正则表达式更易读
match = re.search(pattern, text)

if match:
    print("Match found.")
    bbox = match.group("bbox")
    bev = match.group("bev")
    threed = match.group("threed")
    aos = match.group("aos")

    print(f"bbox AP: {bbox}")
    print(f"bev AP: {bev}")
    print(f"3d AP: {threed}")
    print(f"aos AP: {aos}")
else:
    print("No match found.")

In [ ]:
import os
import re
import json
from collections import defaultdict


CARLA_EVAL_ROOT_PATH = "output/train/CarLA_eval"


def extract_info_from_folder(folder_name):
    # 使用正则表达式提取信息
    match = re.match(
        r"(misrecognize_9|misrecognize_10|physicaladv|vanilla)_([a-zA-Z_]+)",
        folder_name,
    )
    if match:
        attack_method = match.group(1)
        model_name = match.group(2)
        return attack_method, model_name
    return None


def read_evaluation_result_json(folder_path):
    json_path = os.path.join(folder_path, "evaluation_result.json")
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            return json.load(f)
    return None


def get_folder_info_by_model(directory):
    model_dict = defaultdict(lambda: defaultdict(dict))

    # 遍历指定目录中的文件夹
    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)

        if os.path.isdir(folder_path):
            extracted_info = extract_info_from_folder(folder_name)
            print(extracted_info)
            if extracted_info:
                attack_method, model_name = extracted_info
                # 读取 evaluation_result.json 文件
                evaluation_result = read_evaluation_result_json(folder_path)
                if evaluation_result is None:
                    continue
                # 按照模型名称和参数归类
                model_dict[model_name][attack_method] = {
                    "evaluation_result": evaluation_result
                }

    return model_dict


model_dict = get_folder_info_by_model(CARLA_EVAL_ROOT_PATH)
# print(model_dict.keys())
print(json.dumps(model_dict, indent=4))

for attack_method in ["vanilla", "physicaladv", "misrecognize_10", "misrecognize_9"]:
    for model_name in [
        "pointpillar",
        "pointrcnn",
        "pvrcnn",
        "voxel_rcnn_car",
        "second",
    ]:
        car_bev = (
            100.0
            - model_dict[model_name][attack_method]["evaluation_result"]["Car_bev"][0]
        )
        car_3d = (
            100.0
            - model_dict[model_name][attack_method]["evaluation_result"]["Car_3d"][0]
        )
        print(f"& \t{car_bev:.2f}\%& \t{car_3d:.2f}\% ", end=" ")

    print("\\\\")

In [ ]:
import pickle as pkl

# car_valid_rooftop = None
# with open("data/rooftop_approximation/rooftop_appro_kitti.pkl", "rb") as input_stream:
#     car_valid_rooftop = pkl.load(input_stream)
car_count = 0
for li in car_valid_rooftop:
    car_count += sum(1 for item in li if item is not None)
print(car_count)


In [ ]:
import torch
import numpy as np
from data_tools import adv_dataset
from attack_utils import *

from typing import Union
from shapely.geometry import Polygon
from shapely.ops import unary_union


def compute_projected_area_with_overlap(
    vertices: Union[np.ndarray, torch.Tensor], faces: Union[np.ndarray, torch.Tensor]
):
    polygons = []
    if isinstance(vertices, torch.Tensor):
        vertices = vertices.detach().cpu().numpy()
    if isinstance(faces, torch.Tensor):
        faces = faces.detach().cpu().numpy()

    for face in faces:
        # 获取三角形三个顶点的 x-y 坐标
        triangle = vertices[face][:, :2]
        polygon = Polygon(triangle)
        polygons.append(polygon)

    # 3. 合并所有三角形为一个或多个不重叠的多边形
    merged_polygon = unary_union(polygons)

    # 4. 计算合并后的面积
    return merged_polygon.area

In [ ]:
import os
import json


def load_meshes_ckpt(root_dir):
    results = {}

    # 遍历模型名称文件夹
    for model_name in os.listdir(root_dir):
        model_path = os.path.join(root_dir, model_name)
        if os.path.isdir(model_path):
            results[model_name] = {}

            # 遍历攻击方法文件夹
            for attack_method in os.listdir(model_path):
                attack_path = os.path.join(model_path, attack_method)
                if os.path.isdir(attack_path):
                    ckpt_file = os.path.join(
                        attack_path, "final_adversarial_patch_checkpoint.pt"
                    )
                    state_dict = torch.load(ckpt_file)
                    universal_adv_patch_car = single_sphere(
                        scale=adv_dataset.CAR_ADV_PATCH_SCALE
                    )
                    universal_adv_patch_car.load_parameter(
                        state_dict["universal_adv_patch_car"]
                    )
                    adv_meshes = universal_adv_patch_car.get_deformed_meshes()
                    area = compute_projected_area_with_overlap(
                        adv_meshes.verts_packed().detach().cpu().numpy(),
                        adv_meshes.faces_packed().detach().cpu().numpy(),
                    )
                    results[model_name][attack_method] = area

    return results


# 使用示例
root_directory = "output/train/comparison_table"
meshes_area_results = load_meshes_ckpt(root_directory)

vanilla_adv_patch_car = single_sphere(scale=adv_dataset.CAR_ADV_PATCH_SCALE)
vanilla_meshes = vanilla_adv_patch_car.get_deformed_meshes()
meshes_area = compute_projected_area_with_overlap(
    vanilla_meshes.verts_packed().detach().cpu().numpy(),
    vanilla_meshes.faces_packed().detach().cpu().numpy(),
)
print(f"vanilla: {meshes_area:.2f}")

for attack_method in ["misrecognize_10", "misrecognize_9"]:
    for i, model_name in enumerate(
        ["pointpillar", "pointrcnn", "pvrcnn", "voxel_rcnn_car", "second"]
    ):
        meshes_area = meshes_area_results[model_name][attack_method]
        # print(f"{model_name} ({attack_method}): {meshes_area:.2f}")
        print(f"&\t{meshes_area} ", end="")
    print("\\")

In [ ]:
import os
import json


def load_evaluation_results(root_dir):
    results = {}

    # 遍历模型名称文件夹
    for model_name in os.listdir(root_dir):
        model_path = os.path.join(root_dir, model_name)
        if os.path.isdir(model_path):
            results[model_name] = {}

            # 遍历攻击方法文件夹
            for attack_method in os.listdir(model_path):
                attack_path = os.path.join(model_path, attack_method)
                if os.path.isdir(attack_path):
                    evaluation_file = os.path.join(
                        attack_path, "evaluation_result.json"
                    )

                    # 读取evaluation_result.json文件
                    if os.path.exists(evaluation_file):
                        with open(evaluation_file, "r") as f:
                            try:
                                evaluation_data = json.load(f)
                                results[model_name][attack_method] = evaluation_data
                            except json.JSONDecodeError:
                                print(f"Error decoding JSON in file: {evaluation_file}")

    return results


# 使用示例
root_directory = "output/train/comparison_table"
evaluation_results = load_evaluation_results(root_directory)

print(json.dumps(evaluation_results, indent=4))
clean_data = [86.76, 77.57, 85.67, 78.67, 88.92, 84.37, 89.15, 85.66, 88.72, 81.82]
for attack_method in ["misrecognize_10", "misrecognize_9"]:
    for i, model_name in enumerate(
        ["pointpillar", "pointrcnn", "pvrcnn", "voxel_rcnn_car", "second"]
    ):
        model_metrics = evaluation_results[model_name][attack_method]
        clean_bev = clean_data[2 * i]
        clean_3d = clean_data[2 * i + 1]
        car_bev = (clean_bev - model_metrics["Car_bev/moderate"]) / clean_bev * 100
        car_3d = (clean_3d - model_metrics["Car_3d/moderate"]) / clean_3d * 100
        print(f"&\t{car_bev:.2f}\% &\t{car_3d:.2f}\% ", end="")
    print()


In [ ]:
import os
import json
from copy import deepcopy


def load_evaluation_results(root_dir):
    results = {}

    # 遍历模型名称文件夹
    for model_name in os.listdir(root_dir):
        model_path = os.path.join(root_dir, model_name)
        if os.path.isdir(model_path):
            results[model_name] = {}

            # 遍历攻击方法文件夹
            for attack_method in os.listdir(model_path):
                attack_path = os.path.join(model_path, attack_method)
                if os.path.isdir(attack_path):
                    ckpt_file = os.path.join(
                        attack_path, "final_adversarial_patch_checkpoint.pt"
                    )
                    state_dict = torch.load(ckpt_file)
                    universal_adv_patch_car = single_sphere(
                        scale=adv_dataset.CAR_ADV_PATCH_SCALE
                    )
                    universal_adv_patch_car.load_parameter(
                        state_dict["universal_adv_patch_car"]
                    )
                    adv_meshes = universal_adv_patch_car.get_deformed_meshes()
                    area = compute_projected_area_with_overlap(
                        adv_meshes.verts_packed().detach().cpu().numpy(),
                        adv_meshes.faces_packed().detach().cpu().numpy(),
                    )

                    evaluation_file = os.path.join(
                        attack_path, "evaluation_result.json"
                    )
                    if os.path.exists(evaluation_file):
                        with open(evaluation_file, "r") as f:
                            try:
                                evaluation_data = json.load(f)
                                results[model_name][attack_method] = evaluation_data
                            except json.JSONDecodeError:
                                print(f"Error decoding JSON in file: {evaluation_file}")
                    results[model_name][attack_method]["Meshes_BEVArea"] = area

    return results


# 使用示例
root_directory = "output/train/loss_ablation"
evaluation_results = load_evaluation_results(root_directory)
evaluation_results_metric = {}
print(evaluation_results.keys())
# 打印结果
# print(json.dumps(evaluation_results, indent=4))
clean_data = [86.76, 77.57, 85.67, 78.67, 85.67, 78.67, 89.15, 85.66, 89.15, 85.66]
for attack_method in [
    "mislocalize_4",
    "mislocalize_7",
    "mislocalize_9",
    "misrecognize_4",
    "misrecognize_8",
    "misrecognize_9",
    "misrecognize_10",
    "comprehensive_4",
    "comprehensive_9",
]:
    for idx, model_name in enumerate(
        [
            "pointpillar",
            "pointrcnn_s1",
            "pointrcnn",
            "voxel_rcnn_car_s1",
            "voxel_rcnn_car",
        ]
    ):
        clean_bev = clean_data[idx * 2]
        clean_3d = clean_data[idx * 2 + 1]
        car_bev = (
            (
                clean_bev
                - evaluation_results[model_name][attack_method]["Car_bev/moderate"]
            )
            / clean_bev
            * 100
        )
        car_3d = (
            (
                clean_3d
                - evaluation_results[model_name][attack_method]["Car_3d/moderate"]
            )
            / clean_3d
            * 100
        )
        car_bev_area = evaluation_results[model_name][attack_method]["Meshes_BEVArea"]
        if model_name not in evaluation_results_metric:
            evaluation_results_metric[model_name] = {}

        evaluation_results_metric[model_name][attack_method] = {
            "CAR_BEV": car_bev,
            "CAR_3D": car_3d,
            "Meshes_BEV_Area": car_bev_area,
        }
        # print(f",\t{car_bev:.2f}\% ,\t{car_3d:.2f}\% ", end = '')

print(evaluation_results_metric)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

data_x = []
data_y = []
data_cat = []

for model_name, sub_dict in evaluation_results_metric.items():
    for attack_method, eval_dict in sub_dict.items():
        data_x.append(eval_dict["Meshes_BEV_Area"])
        data_y.append(eval_dict["CAR_BEV"])
        data_cat.append(model_name)

sns.scatterplot(
    x=data_x,
    y=data_y,
    hue=data_cat,
    # palette={'A': 'red', 'B': 'green', 'C': 'blue'},
)
plt.title("BEV area vs. ASR in white box attack")
plt.legend(title="Victim model")
plt.show()

In [ ]:
import os
import re
import json
from collections import defaultdict


LEVEL_ABLATION_ROOT_PATH = "output/train/transfer_eval"

# PointRCNN PointPillar PVRCNN VoxelRCNN SECOND
clean_data = [86.76, 77.57, 85.67, 78.67, 88.92, 84.37, 89.15, 85.66, 88.72, 81.82]


def extract_info_from_folder(folder_name):
    # 使用正则表达式提取信息
    match = re.match(r"([a-zA-Z_1]+)_([a-zA-Z]+_[\d]+)", folder_name)
    if match:
        surrogate_model = match.group(1)
        attack_method = match.group(2)
        return surrogate_model, attack_method
    return None


def read_evaluation_result_json(folder_path):
    json_path = os.path.join(folder_path, "evaluation_result.json")
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            return json.load(f)
    return None


def get_folder_info_by_model(directory):
    model_dict = defaultdict(lambda: defaultdict(dict))

    # 遍历指定目录中的文件夹
    for folder_name in os.listdir(directory):
        folder_path = os.path.join(directory, folder_name)
        model_name = folder_name
        for sub_folder_name in os.listdir(folder_path):
            sub_folder_path = os.path.join(folder_path, sub_folder_name)
            if os.path.isdir(sub_folder_path):
                extracted_info = extract_info_from_folder(sub_folder_name)
            if extracted_info:
                surrogate_model, attack_method = extracted_info
                # 读取 evaluation_result.json 文件
                evaluation_result = read_evaluation_result_json(sub_folder_path)
                if evaluation_result is None:
                    continue

                ckpt_file = os.path.join(
                    f"output/train/loss_ablation/{surrogate_model}/{attack_method}",
                    "final_adversarial_patch_checkpoint.pt",
                )
                state_dict = torch.load(ckpt_file)
                universal_adv_patch_car = single_sphere(
                    scale=adv_dataset.CAR_ADV_PATCH_SCALE
                )
                universal_adv_patch_car.load_parameter(
                    state_dict["universal_adv_patch_car"]
                )
                adv_meshes = universal_adv_patch_car.get_deformed_meshes()
                area = compute_projected_area_with_overlap(
                    adv_meshes.verts_packed().detach().cpu().numpy(),
                    adv_meshes.faces_packed().detach().cpu().numpy(),
                )

                model_dict[model_name][surrogate_model][attack_method] = {
                    "evaluation_result": evaluation_result,
                    "CAR_BEV": area,
                }

    return model_dict


model_dict = get_folder_info_by_model(LEVEL_ABLATION_ROOT_PATH)
evaluation_results_metric = defaultdict(lambda: defaultdict(dict))

for surrogate_model in [
    "pointpillar",
    "pointrcnn_s1",
    "pointrcnn",
    "voxel_rcnn_car_s1",
    "voxel_rcnn_car",
]:
    for attack_method in ["misrecognize_9", "misrecognize_10"]:
        for i, victim_model in enumerate(
            ["pointpillar", "pointrcnn", "pvrcnn", "voxel_rcnn_car", "second"]
        ):
            # print("BEV:")
            clean_bev = clean_data[2 * i]
            clean_3d = clean_data[2 * i + 1]
            if surrogate_model not in model_dict[victim_model]:
                continue
            car_bev = (
                (
                    clean_bev
                    - model_dict[victim_model][surrogate_model][attack_method][
                        "evaluation_result"
                    ]["Car_bev/moderate"]
                )
                / clean_bev
                * 100
            )
            car_3d = (
                (
                    clean_3d
                    - model_dict[victim_model][surrogate_model][attack_method][
                        "evaluation_result"
                    ]["Car_3d/moderate"]
                )
                / clean_3d
                * 100
            )
            car_bev_area = model_dict[victim_model][surrogate_model][attack_method][
                "CAR_BEV"
            ]
            evaluation_results_metric[surrogate_model][victim_model][attack_method] = {
                "CAR_BEV": car_bev,
                "CAR_3D": car_3d,
                "Meshes_BEV_Area": car_bev_area,
            }

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

data_x = []
data_y = []
data_cat = []

for surrogate_model, sub_dict in evaluation_results_metric.items():
    for victim_model, dsub_dict in sub_dict.items():
        for attack_method, eval_dict in dsub_dict.items():
            data_x.append(eval_dict["Meshes_BEV_Area"])
            data_y.append(eval_dict["CAR_BEV"])
            data_cat.append(victim_model)

sns.scatterplot(
    x=data_x,
    y=data_y,
    hue=data_cat,
    # palette={'A': 'red', 'B': 'green', 'C': 'blue'},
)
plt.legend(title="Victim model")
plt.title("BEV area vs. ASR in transferability attack")
plt.legend(title="Victim model")
plt.show()